In [49]:

# PHASE 7 — Customer Risk Scoring & Retention Strategy
# Item 21: Generate Customer Risk Scores

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score

df = pd.read_csv("../data/phase6_modeling_data.csv")

# The raw data already has a column called churn_probability (it came with the dataset).
# Rename it so it isn't confused with the score our model will produce.
df = df.rename(columns={"churn_probability": "source_churn_probability"})

features = ['days_since_last_tnx', 'avg_amount_change_pct', 'withdrawal_share_shift',
            'age', 'gender', 'income_bracket', 'occupation', 'education_level',
            'marital_status', 'household_size', 'acquisition_channel', 'savings_account',
            'credit_card', 'personal_loan', 'investment_account', 'insurance_product',
            'satisfaction_score', 'support_tickets_count', 'failed_transactions',
            'app_logins_frequency']

X = pd.get_dummies(df[features], drop_first=True)
y = df['churned']

print(X.shape)
print("Is customer_id unique?", df['customer_id'].is_unique)
print("AUC of the dataset's own churn_probability:",
      roc_auc_score(y, df['source_churn_probability']))


(48723, 62)
Is customer_id unique? True
AUC of the dataset's own churn_probability: 0.498313844822305


In [50]:
# Score every customer the right way
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

# WRONG way (for comparison only): train on everyone, then score the same customers
rf.fit(X, y)
prob_in_sample = rf.predict_proba(X)[:, 1]
print("In-sample AUC (memorized):", roc_auc_score(y, prob_in_sample))

# RIGHT way: out-of-fold predictions with 5-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
churn_probability = cross_val_predict(rf, X, y, cv=cv, method="predict_proba")[:, 1]
print("Out-of-fold AUC (honest):", roc_auc_score(y, churn_probability))

print(pd.Series(churn_probability).describe())


In-sample AUC (memorized): 1.0
Out-of-fold AUC (honest): 0.7445036824749729
count    48723.000000
mean         0.214850
std          0.144056
min          0.003333
25%          0.120000
50%          0.186667
75%          0.270000
max          0.960000
dtype: float64


In [51]:
# Risk Categories
df['churn_probability'] = churn_probability

df['risk_category'] = pd.cut(
    df['churn_probability'],
    bins=[0, 0.2, 0.3, 1.0],
    labels=['Low (<0.20)', 'Medium (0.20-0.30)', 'High (> 0.30)'],
    right=False,            # a bin includes its left edge: 0.2 → Medium, 0.3 → High
    include_lowest=True
)

# Validate the tiers: does actual churn rise from Low → Medium → High?
tier_check = df.groupby('risk_category', observed=True).agg(
    customers=('customer_id', 'count'),
    share=('customer_id', lambda s: len(s) / len(df)),
    avg_probability=('churn_probability', 'mean'),
    actual_churn_rate=('churned', 'mean')
).round(3)
tier_check


,customers,share,avg_probability,actual_churn_rate
risk_category,,,,
Low (<0.20),26055,0.535,0.122,0.103
Medium (0.20-0.30),13782,0.283,0.244,0.241
High (> 0.30),8886,0.182,0.442,0.480


In [52]:
# Customer Value

df['customer_value']= df['customer_lifetime_value']

print(df['customer_value'].describe())

# Is value related to risk?
print("Correlation ( risk vs value):",
      np.corrcoef(df['churn_probability'],df['customer_value'])[0,1])
print(df.groupby('risk_category', observed=True) ['customer_value'].median())

count    4.872300e+04
mean     3.281597e+08
std      7.926224e+08
min      0.000000e+00
25%      5.841438e+07
50%      1.296820e+08
75%      2.973559e+08
max      1.776860e+10
Name: customer_value, dtype: float64
Correlation ( risk vs value): -0.19380673152413286
risk_category
Low (<0.20)           1.652943e+08
Medium (0.20-0.30)    1.022591e+08
High (> 0.30)         9.691769e+07
Name: customer_value, dtype: float64


In [53]:
# The risk table

risk_table = df[['customer_id', 'churn_probability', 'risk_category',
                'customer_value', 'churned']].copy()

risk_table = risk_table.sort_values('churn_probability', ascending=False)

print(risk_table.shape)

risk_table.to_csv("../data/phase7_risk_scores.csv", index=False)
print("Saved.")

risk_table.head(100)


(48723, 5)
Saved.


,customer_id,churn_probability,risk_category,customer_value,churned
42597,74563824,0.960000,High (> 0.30),1.621668e+08,True
16436,58633341,0.956667,High (> 0.30),2.521818e+08,True
27496,37428945,0.953333,High (> 0.30),1.394444e+08,True
19100,68601564,0.953333,High (> 0.30),1.077407e+07,True
43136,80725561,0.950000,High (> 0.30),2.337009e+08,True
...,...,...,...,...,...
5219,13383226,0.903333,High (> 0.30),4.777197e+08,True
40578,77441924,0.903333,High (> 0.30),5.615519e+07,True
1679,98439441,0.903333,High (> 0.30),2.828636e+07,True
15306,39756110,0.903333,High (> 0.30),2.200335e+08,True


In [54]:
## Item 22: Prioritize Customers

# 22.1 Value tier: split customers at the median customer value
median_value = risk_table['customer_value'].median()
print(f"Median customer value: {median_value:,.0f}")

risk_table['value_tier'] = np.where(
    risk_table['customer_value'] >= median_value, 'High Value', 'Low Value'
)

risk_table['value_tier'].value_counts()


Median customer value: 129,681,954


value_tier
High Value    24362
Low Value     24361
Name: count, dtype: int64

In [55]:
# the risk and value matrix

# 22.2a How many customers fall into each cell?
matrix_counts = pd.crosstab(risk_table['risk_category'], risk_table['value_tier'],
                            margins=True)
print("Number of customers:")
print(matrix_counts)

# 22.2b What share of each cell actually churned?
matrix_churn = pd.crosstab(risk_table['risk_category'], risk_table['value_tier'],
                           values=risk_table['churned'].astype(int),
                           aggfunc='mean').round(3)
print("\nActual churn rate:")
print(matrix_churn)


Number of customers:
value_tier          High Value  Low Value    All
risk_category                                   
Low (<0.20)              14936      11119  26055
Medium (0.20-0.30)        5769       8013  13782
High (> 0.30)             3657       5229   8886
All                      24362      24361  48723

Actual churn rate:
value_tier          High Value  Low Value
risk_category                            
Low (<0.20)              0.080      0.134
Medium (0.20-0.30)       0.213      0.262
High (> 0.30)            0.426      0.517


In [56]:
# 22.3 Expected value at risk per customer
risk_table['expected_value_at_risk'] = (
    risk_table['churn_probability'] * risk_table['customer_value']
)

matrix_value_at_risk = pd.crosstab(risk_table['risk_category'], risk_table['value_tier'],
                                   values=risk_table['expected_value_at_risk'],
                                   aggfunc='sum')

print("Share of total expected value at risk:")
(matrix_value_at_risk / matrix_value_at_risk.values.sum()).round(3)


Share of total expected value at risk:


value_tier,High Value,Low Value
risk_category,,
Low (<0.20),0.450,0.040
Medium (0.20-0.30),0.208,0.051
High (> 0.30),0.193,0.058


In [57]:
# 22.4 Priority segments (these match the outline's item 23 strategies)
risk_level = risk_table['risk_category'].astype(str).str.split().str[0]   # "High (> 0.30)" → "High"

conditions = [
    (risk_level == 'High') & (risk_table['value_tier'] == 'High Value'),
    (risk_level == 'High') & (risk_table['value_tier'] == 'Low Value'),
    (risk_level == 'Medium'),
    (risk_level == 'Low'),
]
segments = [
    '1. High Risk + High Value',
    '2. High Risk + Low Value',
    '3. Medium Risk',
    '4. Low Risk',
]
risk_table['priority_segment'] = np.select(conditions, segments, default='Unassigned')

# Rank: segment 1 first; within each segment, the most money at risk first
risk_table = risk_table.sort_values(['priority_segment', 'expected_value_at_risk'],
                                    ascending=[True, False])
risk_table['priority_rank'] = range(1, len(risk_table) + 1)

# Summary of each segment
segment_summary = risk_table.groupby('priority_segment').agg(
    customers=('customer_id', 'count'),
    actual_churn_rate=('churned', 'mean'),
    median_value=('customer_value', 'median'),
    expected_value_at_risk=('expected_value_at_risk', 'sum'),
)
segment_summary['share_of_value_at_risk'] = (
    segment_summary['expected_value_at_risk'] / segment_summary['expected_value_at_risk'].sum()
)
segment_summary.round(3)


,customers,actual_churn_rate,median_value,expected_value_at_risk,share_of_value_at_risk
priority_segment,,,,,
1. High Risk + High Value,3657,0.426,2.331372e+08,4.559506e+11,0.193
2. High Risk + Low Value,5229,0.517,5.553528e+07,1.363731e+11,0.058
3. Medium Risk,13782,0.241,1.022591e+08,6.093544e+11,0.259
4. Low Risk,26055,0.103,1.652943e+08,1.155368e+12,0.490


In [58]:
# 22.5 The first 10 customers the retention team should contact
risk_table[['priority_rank', 'customer_id', 'priority_segment', 'churn_probability',
            'customer_value', 'expected_value_at_risk', 'churned']].head(100)


,priority_rank,customer_id,priority_segment,churn_probability,customer_value,expected_value_at_risk,churned
5995,1,16662064,1. High Risk + High Value,0.306667,2.273774e+09,6.972908e+08,False
43586,2,98290389,1. High Risk + High Value,0.373333,1.823065e+09,6.806110e+08,False
8812,3,38820898,1. High Risk + High Value,0.810000,8.183317e+08,6.628487e+08,True
13917,4,68588216,1. High Risk + High Value,0.513333,1.263876e+09,6.487898e+08,True
36262,5,14384422,1. High Risk + High Value,0.763333,8.436437e+08,6.439813e+08,True
...,...,...,...,...,...,...,...
46819,96,58846280,1. High Risk + High Value,0.576667,6.344218e+08,3.658499e+08,False
4882,97,71709415,1. High Risk + High Value,0.480000,7.619201e+08,3.657217e+08,False
32566,98,74143182,1. High Risk + High Value,0.306667,1.191480e+09,3.653871e+08,False
11378,99,52197305,1. High Risk + High Value,0.396667,9.151893e+08,3.630251e+08,True


In [59]:
# 22.6 Save: overwrites the item 21 file, now with the new columns added
risk_table.to_csv("../data/phase7_risk_scores.csv", index=False)
print(risk_table.shape)
print("Saved.")


(48723, 9)
Saved.


### Item 21 Summary — Customer Risk Scores

**Approach**

- The dataset's provided `churn_probability` column had **no predictive value (AUC 0.50)**, so it was renamed `source_churn_probability` and not used.
- Every customer was scored with the Random Forest from Phase 6, using the same 62 features.
- Scores were generated with **5-fold out-of-fold cross-validation**, so each customer was scored by a model that never saw their outcome. Scoring the training data directly gives a memorized AUC of 1.0; the honest out-of-fold AUC is **0.745**, consistent with the Phase 6 test AUC of 0.761.

**Risk tiers**

| Risk category | Customers | Share | Avg predicted probability | Actual churn rate |
|---|---|---|---|---|
| Low (< 0.20) | 26,055 | 53.5% | 12.2% | **10.3%** |
| Medium (0.20–0.30) | 13,782 | 28.3% | 24.4% | **24.1%** |
| High (≥ 0.30) | 8,886 | 18.2% | 44.2% | **48.0%** |

**Key findings**

- **The tiers are meaningful.** High-risk customers churned at 48%, **4.7× the rate** of low-risk customers.
- **The probabilities are well calibrated.** Predicted and actual churn are within ~4 points in every tier, so scores can be read as real likelihoods.
- **The model's most confident alerts are reliable.** All of the top 100 highest-risk customers actually churned, as did 96.9% of the top 1,000.
- **Customer value is highly concentrated.** The top 10% of customers hold 56% of total lifetime value, and value is weakly negatively correlated with risk (r = −0.19): riskier customers tend to be worth somewhat less.

**Output:** `phase7_risk_scores.csv` with `customer_id`, `churn_probability`, `risk_category` and `customer_value` for all 48,723 customers.


### Item 22 Summary — Customer Prioritization

**Approach**

- **Value tier:** customers at or above the median lifetime value (~130M COP, about $30K) are labeled High Value; the rest are Low Value.
- **Expected value at risk** = churn probability × customer value: the expected loss for each customer if no action is taken.
- Customers were grouped into the four action segments from the retention strategy and ranked by expected value at risk within each segment (`priority_rank`).

**Priority segments**

| Segment | Customers | Actual churn | Median value | Share of value actually lost* |
|---|---|---|---|---|
| **1. High Risk + High Value** | 3,657 (7.5%) | 42.6% | 233M | **26.6%** |
| 2. High Risk + Low Value | 5,229 (10.7%) | **51.7%** | 56M | 9.6% |
| 3. Medium Risk | 13,782 (28.3%) | 24.1% | 102M | 30.7% |
| 4. Low Risk | 26,055 (53.5%) | 10.3% | 165M | 33.0% |

*Share of total lifetime value belonging to customers who actually churned.

**Key findings**

- **Segment 1 is small but costly.** 7.5% of customers account for about 27% of the lifetime value lost to churn, which justifies personal, higher-cost retention efforts.
- **Segment 2 has the highest churn rate (52%) but low value per customer**, so it calls for low-cost, automated retention.
- **Within every risk tier, low-value customers churn 5–9 points more** than high-value customers.
- **Ranking by risk × value changes who comes first.** The #1 priority customer has only a 31% churn probability but is worth 2.27 billion COP (about $525K). The top 100 by risk alone all churned, and 43% of the top 100 by value at risk did, but each of them represents far more value.

**Limitation**

The model does not use customer value, and it **overestimates churn risk for the most valuable customers**: 12.3% predicted vs 4.9% actual in the top value decile. As a result, expected value at risk is inflated for very high-value, low-risk customers (45% of the estimated total, compared with a much smaller share of the value actually lost). Before an expensive intervention, the retention team should confirm each customer's actual warning signals.

**Output:** `phase7_risk_scores.csv` updated with `value_tier`, `expected_value_at_risk`, `priority_segment` and `priority_rank`. This is the input for the Phase 8 Tableau dashboards.


In [60]:
# Validation: share of lifetime value actually lost (customers who really churned), by segment
risk_table['value_lost'] = risk_table['customer_value'] * risk_table['churned']
lost = risk_table.groupby('priority_segment')['value_lost'].sum()
print((lost / lost.sum()).round(3))

# Hit rate of the top 1,000 by churn probability
print("Top 1,000 by risk — actual churn rate:",
      risk_table.nlargest(1000, 'churn_probability')['churned'].mean())


priority_segment
1. High Risk + High Value    0.266
2. High Risk + Low Value     0.096
3. Medium Risk               0.307
4. Low Risk                  0.330
Name: value_lost, dtype: float64
Top 1,000 by risk — actual churn rate: 0.969


In [62]:
## Item 23: Design Intervention Strategies

# 23.1 Map each priority segment to its retention action (from the outline)
action_map = {
    '1. High Risk + High Value': 'Personalized customer-success outreach',
    '2. High Risk + Low Value':  'Automated retention campaign',
    '3. Medium Risk':            'Targeted engagement campaign',
    '4. Low Risk':               'Continue monitoring',
}
risk_table['recommended_action'] = risk_table['priority_segment'].map(action_map)

# Check: every customer has an action, and each segment maps to exactly one
print("Missing actions:", risk_table['recommended_action'].isna().sum())
print(risk_table.groupby(['priority_segment', 'recommended_action']).size())

# Save the final Phase 7 table (drop value_lost if you created it — it uses the outcome)
final_table = risk_table.drop(columns=['value_lost'], errors='ignore')
final_table.to_csv("../data/phase7_risk_scores.csv", index=False)
print(final_table.shape)
print("Saved.")


Missing actions: 0
priority_segment           recommended_action                    
1. High Risk + High Value  Personalized customer-success outreach     3657
2. High Risk + Low Value   Automated retention campaign               5229
3. Medium Risk             Targeted engagement campaign              13782
4. Low Risk                Continue monitoring                       26055
dtype: int64
(48723, 10)
Saved.
